# LAB 1

### Downloading file - Global Energy Transition & Renewables (1900-2026)

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dbr_dev.gabrielajaniszews786.landing;

In [0]:
import requests
url = 'https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv'
file_Path = '/Volumes/dbr_dev/gabrielajaniszews786/landing/owid-energy-data.csv'
r = requests.get(url)
if r.status_code == 200:
    with open(file_Path, 'wb') as f:
        f.write(r.content)
    print('File downloaded successfully')
else:
    print('Error downloading file')


In [0]:
# Reading the data
df = spark.read.csv(file_Path, header=True, inferSchema=True)

df.show(10)

In [0]:
# Checking the schema
df.printSchema()

### Loading the data into a Delta table

In [0]:
# Loading the data into a Delta table
login = 'gabrielajaniszews786'
schema = f'dbr_dev.{login}'
table = f'{schema}.owid_energy'

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

df.write.format("delta").mode("overwrite").saveAsTable(table)

### Basic operations on the DataFrame

In [0]:
from pyspark.sql import functions as F

# Selecting shares of various types of energy for analysis
df_sel = df.select("country", "year", "iso_code", "gas_share_energy", "coal_share_energy", "hydro_share_energy", "nuclear_share_energy", "biofuel_share_energy", "solar_share_energy", "wind_share_energy", "other_renewables_share_energy","renewables_share_energy", "fossil_share_energy", "low_carbon_share_energy", "gdp")

df_sel.show(5)

In [0]:
# Selecting last decade and countries with no aggregates (i.e. ASEAN)
df_filtered = df_sel.filter(
    (F.col("year") >= 2015) &
    (F.col("iso_code").isNotNull())
)
df_filtered.show(5)


In [0]:
# Checking the last year with available renewables energy data
df_filtered.filter(F.col("renewables_share_energy").isNotNull()).agg(F.max("year")).show()

In [0]:
# Top 10 countries with the highest renewables energy share in the last year
df_filtered.filter((F.col("year") == 2024)).orderBy(F.desc("renewables_share_energy")).limit(10).show()


### Using API Countries for regions

In [0]:
import requests
resp = requests.get("https://www.apicountries.com/countries")
countries_json = resp.json()
# Creating a countries_df table with iso_codes, regions and subregions
countries_df = spark.createDataFrame(
    [(c.get("alpha3Code"), c.get("region"), c.get("subregion")) for c in countries_json],
    ["iso_code", "region", "subregion"]
)
countries_df.show(10)

### Joining the main energy table with countries table for regional aggregation

In [0]:
# Joing the tables on iso_code
df_joined = df_filtered.join(countries_df, on="iso_code", how="left")
df_joined.show(5)


In [0]:
# Renewables energy share by regions of the world
df_joined.groupBy("region").agg(F.avg("renewables_share_energy").alias("renewables_share_energy")).orderBy(F.desc("renewables_share_energy")).show(20)

In [0]:
# Checking if there are any countries with no region
df_joined.filter(F.col("region").isNull()).select("country", "iso_code").distinct().show(50, truncate=False)

In [0]:
# Removing Netherlands Antilles from the data as this country no longer exists
df_joined = df_joined.filter(F.col("region").isNotNull())

In [0]:
# Formatting the GDP column for better visualization, leaving the raw column
df_joined = df_joined.withColumn("gdp_billion_usd", F.col("gdp") / 1e9)

### Saving the enriched table for the dashboard

In [0]:
# Writing the enriched table, mergeSchema due to the fact that the original table had fewer columns
df_joined.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{schema}.owid_energy_enriched")